# Tiny ImageNet - Model Extraction Defense Test

Runs the model-extraction **defense sweep** with **Tiny ImageNet** as the
attacker's out-of-distribution transfer (query) pool, and appends the results
(`transfer=tinyimagenet`) to a defense-results CSV, alongside the existing
`cifar100` rows.

- The victim (`checkpoints/victim.pt`, ResNet-18) ships inside the repo.
- Tiny ImageNet is only the attacker's **query pool**; scoring (accuracy /
  fidelity) is always on the clean **CIFAR-10 test** split, like the existing rows.
- Run top to bottom on a **GPU runtime** (Colab / Kaggle). The sweep auto-selects
  CUDA when available.

**Resumable.** Each finished `(defense, budget)` row is written to the output CSV
immediately, and a re-run skips rows already present. If you keep the output on
**Google Drive** (step 3), progress survives a recycled runtime: just re-run all
cells and it continues where it stopped.


## 1. Clone the repository

In [ ]:
# Gets the repo (includes the trained victim checkpoint). data/ is .gitignored and
# is downloaded in step 4. Works whether this notebook was uploaded standalone
# (clones the repo) or opened from inside an existing clone (runs in place).
import os
REPO_URL = "https://github.com/azraihan/security_project_model_extraction.git"
REPO_DIR = "security_project_model_extraction"

if os.path.exists("run_tinyimagenet_defense.py"):
    print("already inside the repo; skipping clone")
elif os.path.isdir(REPO_DIR):
    print("repo dir present; entering and pulling latest")
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!git log --oneline -3
print("victim checkpoint present:", os.path.exists("checkpoints/victim.pt"))

## 2. Dependencies + GPU check

In [ ]:
# torch / torchvision / numpy / matplotlib / tqdm / requests / pillow are
# preinstalled on Colab & Kaggle, so we only add the victim-server deps here.
# (Deliberately NOT running `pip install -r requirements.txt` to avoid disturbing
#  the preinstalled, GPU-matched torch build.)
!pip -q install fastapi "uvicorn[standard]"

import torch
dev = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", __import__("torchvision").__version__)
print("selected device:", dev)
if dev == "cpu":
    print("WARNING: no GPU detected - the sweep will be slow. Switch to a GPU runtime.")

## 3. Durable storage (survives a recycled runtime)

In [ ]:
# Put the results CSV (and the Tiny ImageNet cache) on Google Drive so a Colab
# disconnect never loses progress -- re-running then RESUMES instead of restarting.
# Set USE_DRIVE = False to keep everything in the ephemeral repo instead.
USE_DRIVE = True

import os, shutil
POOL_CACHE = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PERSIST = "/content/drive/MyDrive/model_extraction"
        os.makedirs(PERSIST, exist_ok=True)
        OUT_CSV = os.path.join(PERSIST, "defense_results.csv")
        POOL_CACHE = os.path.join(PERSIST, "tinyimagenet_train_32.npy")
        # Seed the durable CSV once from the committed baseline (20 cifar100 rows).
        if not os.path.exists(OUT_CSV):
            shutil.copy("results/defense_results.csv", OUT_CSV)
    except Exception as e:
        print("Drive unavailable (", e, ") -> using ephemeral storage")
        USE_DRIVE = False

if not USE_DRIVE:
    OUT_CSV = "results/defense_results.csv"

print("USE_DRIVE =", USE_DRIVE)
print("OUT_CSV   =", OUT_CSV)

## 4. Fetch the data (into ./data)

In [ ]:
# Downloads/caches, one time:
#   - CIFAR-10 test split  (used ONLY to score accuracy/fidelity)
#   - Tiny ImageNet train pool, resized 64 -> 32, cached as
#     data/tinyimagenet_train_32.npy  (~100k images, a few minutes the first time)
# If a Drive-cached pool exists, it is restored to skip the rebuild.
import os, shutil
os.makedirs("data", exist_ok=True)
LOCAL_POOL = "data/tinyimagenet_train_32.npy"

if POOL_CACHE and os.path.exists(POOL_CACHE) and not os.path.exists(LOCAL_POOL):
    print("restoring Tiny ImageNet cache from Drive...")
    shutil.copy(POOL_CACHE, LOCAL_POOL)

from data import cifar10_eval_arrays, transfer_pool
imgs, y = cifar10_eval_arrays(download=True)
print("cifar10 test:", imgs.shape, y.shape)
pool = transfer_pool("tinyimagenet")   # builds + caches on first call
print("tinyimagenet pool:", pool.shape, pool.dtype)

if POOL_CACHE and not os.path.exists(POOL_CACHE):
    print("saving Tiny ImageNet cache to Drive for next time...")
    shutil.copy(LOCAL_POOL, POOL_CACHE)

## 5. (Optional) Seed the 13 rows already computed in the earlier run

In [ ]:
# The earlier notebook run finished 13 of the 20 rows before Colab disconnected.
# Seeding them here makes the resumable sweep SKIP them and compute only the
# remaining 7 (top-1 prob 10k/20k/50k + all four label_only). Set SEED_PRIOR=False
# to instead recompute all 20 from scratch on this runtime.
SEED_PRIOR = True

import csv
if SEED_PRIOR:
    # victim's CIFAR-10 test accuracy: the same constant evaluate() records per row,
    # computed once here to fill the column the earlier log did not print.
    from evaluate import _predict_all
    from data import cifar10_eval_arrays
    from victim_train import load_victim
    _vic, _ = load_victim("checkpoints/victim.pt", device=dev)
    _imgs, _y = cifar10_eval_arrays(download=False)
    V = round(float((_predict_all(_vic, _imgs, dev) == _y).mean()), 4)
    print("victim test accuracy (this runtime):", V)

    # (defense, queries, substitute_accuracy, fidelity) from the earlier run's log.
    PRIOR = [
        ("none (soft)", 5000, 0.3234, 0.3231), ("none (soft)", 10000, 0.3417, 0.3418),
        ("none (soft)", 20000, 0.5083, 0.5133), ("none (soft)", 50000, 0.7469, 0.7599),
        ("round-1 (soft)", 5000, 0.3266, 0.3277), ("round-1 (soft)", 10000, 0.3816, 0.3835),
        ("round-1 (soft)", 20000, 0.5752, 0.5821), ("round-1 (soft)", 50000, 0.7224, 0.7332),
        ("noise (soft)", 5000, 0.1956, 0.1960), ("noise (soft)", 10000, 0.3331, 0.3314),
        ("noise (soft)", 20000, 0.1526, 0.1512), ("noise (soft)", 50000, 0.6731, 0.6837),
        ("top-1 prob (soft)", 5000, 0.2257, 0.2255),
    ]
    FIELDS = ["defense","transfer","queries","loss","substitute_accuracy","victim_accuracy","fidelity"]
    with open(OUT_CSV, newline="") as f:
        have = {(r["defense"], int(r["queries"]))
                for r in csv.DictReader(f) if r["transfer"] == "tinyimagenet"}
    added = 0
    with open(OUT_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        for d, q, acc, fid in PRIOR:
            if (d, q) in have:
                continue
            w.writerow({"defense": d, "transfer": "tinyimagenet", "queries": q,
                        "loss": "soft", "substitute_accuracy": acc,
                        "victim_accuracy": V, "fidelity": fid})
            added += 1
    print(f"seeded {added} prior rows into {OUT_CSV} (the sweep will skip these)")
else:
    print("SEED_PRIOR=False -> all 20 rows will be recomputed on this runtime")

## 6. Run the Tiny ImageNet defense sweep (resumable)

In [ ]:
# 5 defenses x budgets [5k, 10k, 20k, 50k] x 40 substitute epochs, scored on
# CIFAR-10 test. Auto-selects CUDA. Appends transfer=tinyimagenet rows to OUT_CSV
# and SKIPS any already present -- so this cell is safe to re-run after a
# disconnect; it continues from where it stopped.
!python run_tinyimagenet_defense.py --out "{OUT_CSV}"

## 7. Inspect the results

In [ ]:
import pandas as pd
df = pd.read_csv(OUT_CSV)
print("rows:", len(df), "| transfers:", df["transfer"].unique().tolist())
tin = df[df["transfer"] == "tinyimagenet"].reset_index(drop=True)
print("tinyimagenet rows:", len(tin), "/ 20")
tin

## Notes

- **Resuming after a disconnect:** just re-run cells 1-6. With Drive on, step 4 is
  fast (pool restored from cache) and step 5 skips the rows already in `OUT_CSV`,
  finishing only what's left.
- **Get the final CSV into the repo / downloaded:**
  ```python
  import shutil; shutil.copy(OUT_CSV, "results/defense_results.csv")   # into the clone
  from google.colab import files; files.download(OUT_CSV)              # to your machine
  ```
- **Fast sanity check** (1 defense, tiny budget, ~1 min):
  ```bash
  !python run_tinyimagenet_defense.py --max-plans 1 --budgets 300 --sub-epochs 1 --out /tmp/quick.csv
  ```
